In [ ]:
import duckdb
import pydeck as pdk

con = duckdb.connect("france.duckdb")
con.execute("LOAD spatial")

In [ ]:
con.sql("DESCRIBE anfr_sup_emetteur").df()

In [ ]:
con.sql("DESCRIBE anfr_sup_bande").df()

In [ ]:
con.sql("""
    SELECT EMR_LB_SYSTEME, COUNT(*) AS n
    FROM anfr_sup_emetteur
    GROUP BY 1 ORDER BY 2 DESC
    LIMIT 40
""").df()

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE power_assumptions AS
    SELECT * FROM (VALUES
        ('GSM 900',    0.25),
        ('GSM 1800',   0.25),
        ('UMTS 900',   0.25),
        ('UMTS 2100',  0.25),
        ('LTE 700',    0.30),
        ('LTE 800',    0.30),
        ('LTE 900',    0.30),
        ('LTE 1800',   0.35),
        ('LTE 2100',   0.35),
        ('LTE 2600',   0.35),
        ('5G NR 700',  0.35),
        ('5G NR 2100', 0.40),
        ('5G NR 3500', 1.00)
    ) t(system, kw_per_sector)
""")
SITE_FIXED_KW = 1.0
OVERHEAD = 1.2

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE carriers AS
    WITH st AS (
        SELECT DISTINCT STA_NM_ANFR, SUP_ID, operator, insee, city, lat, lon
        FROM antennas
    )
    SELECT e.EMR_ID, e.STA_NM_ANFR, e.EMR_LB_SYSTEME AS system, e.EMR_DT_SERVICE AS service_date,
           st.SUP_ID, st.operator, st.insee, st.city, st.lat, st.lon, p.kw_per_sector
    FROM anfr_sup_emetteur e
    JOIN power_assumptions p ON e.EMR_LB_SYSTEME = p.system
    JOIN st USING (STA_NM_ANFR)
    WHERE NULLIF(e.EMR_DT_SERVICE, '') IS NOT NULL
""")
con.sql("SELECT operator, system, COUNT(*) AS n FROM carriers GROUP BY 1, 2 ORDER BY 1, 2").df()

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE site_energy AS
    SELECT STA_NM_ANFR, SUP_ID, operator, insee, city, lat, lon,
           COUNT(*) AS n_carriers,
           string_agg(DISTINCT system, ', ') AS systems,
           SUM(kw_per_sector) AS radio_kw,
           (SUM(kw_per_sector) + {SITE_FIXED_KW}) * {OVERHEAD} AS site_kw,
           (SUM(kw_per_sector) + {SITE_FIXED_KW}) * {OVERHEAD} * 8760 AS kwh_year
    FROM carriers
    GROUP BY STA_NM_ANFR, SUP_ID, operator, insee, city, lat, lon
""")

In [ ]:
con.sql("""
    SELECT operator,
           COUNT(*) AS site,
           round(AVG(site_kw), 2) AS ort_kw,
           round(SUM(kwh_year) / 1e9, 2) AS twh_yil
    FROM site_energy
    GROUP BY 1
    UNION ALL
    SELECT 'TOPLAM', COUNT(*), round(AVG(site_kw), 2), round(SUM(kwh_year) / 1e9, 2)
    FROM site_energy
    ORDER BY 1
""").df()

In [ ]:
TARGET_TWH = 2.8

total_twh = con.sql("SELECT SUM(kwh_year) / 1e9 FROM site_energy").fetchone()[0]
K = TARGET_TWH / total_twh
print(f"model: {total_twh:.2f} TWh  hedef: {TARGET_TWH} TWh  K: {K:.3f}")

con.execute(f"""
    CREATE OR REPLACE TABLE site_energy_cal AS
    SELECT *, site_kw * {K} AS site_kw_cal, kwh_year * {K} AS kwh_year_cal
    FROM site_energy
""")

In [ ]:
con.sql("""
    SELECT operator,
           COUNT(*) AS site,
           round(AVG(site_kw_cal), 2) AS ort_kw,
           round(SUM(kwh_year_cal) / 1e9, 3) AS twh_yil,
           round(AVG(n_carriers), 1) AS ort_verici
    FROM site_energy_cal
    GROUP BY 1 ORDER BY 1
""").df()

In [ ]:
con.sql("""
    SELECT a.city, e.operator,
           COUNT(DISTINCT e.STA_NM_ANFR) AS site,
           round(SUM(e.kwh_year_cal) / 1e6, 1) AS gwh_yil,
           round(AVG(e.site_kw_cal), 2) AS ort_kw
    FROM site_energy_cal e
    JOIN (SELECT DISTINCT STA_NM_ANFR, city FROM antennas_area) a USING (STA_NM_ANFR)
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()

In [ ]:
import pandas as pd

def run_scenario(kw3500, fixed, overhead, target_twh=2.8):
    q = f"""
        WITH c AS (
            SELECT STA_NM_ANFR, operator,
                   CASE WHEN system = '5G NR 3500' THEN {kw3500} ELSE kw_per_sector END AS kw
            FROM carriers
        ),
        s AS (
            SELECT STA_NM_ANFR, operator, (SUM(kw) + {fixed}) * {overhead} * 8760 AS kwh
            FROM c GROUP BY 1, 2
        ),
        k AS (SELECT {target_twh} * 1e9 / SUM(kwh) AS K FROM s),
        a AS (SELECT DISTINCT STA_NM_ANFR, city FROM antennas_area)
        SELECT a.city, s.operator,
               AVG(s.kwh * k.K) / 8760 AS ort_kw,
               SUM(s.kwh * k.K) / 1e6 AS gwh
        FROM s JOIN a USING (STA_NM_ANFR) CROSS JOIN k
        GROUP BY 1, 2
    """
    df = con.sql(q).df()
    df["kw3500"], df["fixed"], df["overhead"] = kw3500, fixed, overhead
    return df

In [ ]:
scenarios = [
    (kw, fx, 1.2)
    for kw in [0.6, 1.0, 1.5]
    for fx in [0.5, 1.0, 2.0]
]
res = pd.concat([run_scenario(*s) for s in scenarios], ignore_index=True)
res["senaryo"] = "5G=" + res["kw3500"].astype(str) + " / sabit=" + res["fixed"].astype(str)
res["sira"] = res.groupby(["senaryo", "city"])["ort_kw"].rank(ascending=False).astype(int)

In [ ]:
res.pivot_table(index=["city", "operator"], columns="senaryo", values="sira")

In [ ]:
res.groupby(["city", "operator"])["ort_kw"].agg(["min", "max"]).round(2)

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE mast_energy AS
    SELECT e.SUP_ID, a.city,
           ANY_VALUE(e.lat) AS lat, ANY_VALUE(e.lon) AS lon,
           COUNT(DISTINCT e.operator) AS n_ops,
           string_agg(DISTINCT e.operator, ', ') AS ops,
           SUM(e.site_kw_cal) AS kw,
           SUM(e.kwh_year_cal) / 1e3 AS mwh
    FROM site_energy_cal e
    JOIN (SELECT DISTINCT STA_NM_ANFR, city FROM antennas_area) a USING (STA_NM_ANFR)
    GROUP BY 1, 2
""")
con.sql("SELECT city, COUNT(*) AS direk, round(AVG(kw), 2) AS ort_kw, round(MAX(kw), 1) AS max_kw FROM mast_energy GROUP BY 1").df()

In [ ]:
import numpy as np

BLUE_RED = np.array([
    [33, 102, 172],
    [110, 76, 170],
    [190, 50, 110],
    [215, 25, 28],
])

def color_scale(values, alpha=220):
    lo, hi = values.quantile([0.05, 0.95])
    t = ((values - lo) / (hi - lo)).clip(0, 1)
    pos = np.linspace(0, 1, len(BLUE_RED))
    rgb = np.stack([np.interp(t, pos, BLUE_RED[:, i]) for i in range(3)], axis=1)
    colors = [[int(r), int(g), int(b), int(alpha)] for r, g, b in rgb]
    return colors, t

masts = con.sql("SELECT * FROM mast_energy").df()
masts["color"], t = color_scale(masts["kw"])
masts["radius"] = 20 + 60 * np.sqrt(t)
masts["label"] = ("Direk: " + masts["ops"] + " | " + masts["kw"].round(1).astype(str)
                  + " kW | " + masts["mwh"].round(0).astype(int).astype(str) + " MWh/yıl")

stops = con.sql("SELECT city, stop_name, lat, lon FROM stops_clean").df()
stops["label"] = "Durak: " + stops["stop_name"]

stops_layer = pdk.Layer(
    "ScatterplotLayer", data=stops, get_position=["lon", "lat"],
    get_fill_color=[110, 110, 110, 140],
    get_radius=12, radius_min_pixels=1, radius_max_pixels=3, pickable=True,
)
mast_layer = pdk.Layer(
    "ScatterplotLayer", data=masts, get_position=["lon", "lat"],
    get_fill_color="color", get_radius="radius",
    radius_min_pixels=2, radius_max_pixels=12, pickable=True,
)

deck = pdk.Deck(
    layers=[stops_layer, mast_layer],
    initial_view_state=pdk.ViewState(latitude=48.8566, longitude=2.3522, zoom=11),
    map_style="light",
    tooltip={"text": "{city}\n{label}"},
)
deck.to_html("energy_map.html")
deck
#direk enerji haritası

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE stop_energy AS
    WITH pairs AS (
        SELECT s.city, s.stop_id, s.stop_name, s.lat, s.lon,
               e.STA_NM_ANFR, e.operator, e.site_kw_cal,
               111320 * sqrt(power(e.lat - s.lat, 2) + power((e.lon - s.lon) * cos(radians(s.lat)), 2)) AS dist_m
        FROM stops_clean s
        JOIN site_energy_cal e
          ON e.lat BETWEEN s.lat - 0.0045 AND s.lat + 0.0045
         AND e.lon BETWEEN s.lon - 0.0065 AND s.lon + 0.0065
    )
    SELECT city, stop_id, stop_name, lat, lon,
           COUNT(DISTINCT STA_NM_ANFR) AS n_sites,
           COUNT(DISTINCT operator) AS n_ops,
           SUM(site_kw_cal) AS kw_500m,
           MIN(dist_m) AS nearest_m
    FROM pairs
    WHERE dist_m <= 500
    GROUP BY ALL
""")

In [ ]:
con.sql("""
    SELECT city, stop_name,
           round(MAX(kw_500m), 1) AS kw_500m,
           MAX(n_sites) AS site,
           round(MIN(nearest_m)) AS en_yakin_m
    FROM stop_energy
    GROUP BY 1, 2
    QUALIFY row_number() OVER (PARTITION BY city ORDER BY MAX(kw_500m) DESC) <= 10
    ORDER BY city, kw_500m DESC
""").df()

In [ ]:
se = con.sql("SELECT * FROM stop_energy").df()
se["color"], t = color_scale(se["kw_500m"], alpha=210)
se["label"] = ("Durak: " + se["stop_name"] + " | " + se["kw_500m"].round(1).astype(str)
               + " kW | " + se["n_sites"].astype(str) + " site")

stop_load_layer = pdk.Layer(
    "ScatterplotLayer", data=se, get_position=["lon", "lat"],
    get_fill_color="color", get_radius=40,
    radius_min_pixels=2, radius_max_pixels=8, pickable=True,
)

deck2 = pdk.Deck(
    layers=[stop_load_layer],
    initial_view_state=pdk.ViewState(latitude=48.8566, longitude=2.3522, zoom=11),
    map_style="light",
    tooltip={"text": "{city}\n{label}"},
)
deck2.to_html("stop_load_map.html")
deck2
#durak enerji yükü haritası